In [1]:
# injected-parameters


# AUDIT — Stage 02 integrate (atlas scVI + clustering)
**Read-only.** Sources of truth: `workflow/scripts/integrate.py`, `config/config.yaml`, the paper's
integration paragraph, and — decisively — **the output objects' own records**. Verifies: HVG recipe
(3k, mito/ribo excluded) against the object's var; latent shape; graph params from `uns`; ladder
presence; that the netscratch deliverables' latents are byte-identical row-subsets of these atlas
latents; and shows the stored UMAPs (no retraining anywhere).

In [2]:
import json, glob, os, re, pathlib
import numpy as np, pandas as pd, h5py, yaml
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from anndata.io import read_elem
X = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
CHECKS = []
def check(label, ok, got=None, want=None, warn=False):
    CHECKS.append(dict(label=label, ok=bool(ok), warn=warn, got=str(got)[:160], want=str(want)[:120]))
    print(("PASS " if ok else ("WARN " if warn else "FAIL ")) + f"{label}: got={got}" + (f" want={want}" if want is not None else ""))
def summary(tag):
    bad = [c for c in CHECKS if not c["ok"] and not c["warn"]]
    warns = [c for c in CHECKS if not c["ok"] and c["warn"]]
    json.dump(CHECKS, open(f"audit_{tag}_report.json", "w"), indent=1)
    print(f"\n===== {len(CHECKS)} checks | {len(bad)} FAIL | {len(warns)} WARN =====")
    for c in bad + warns: print(("FAIL " if not c["warn"] else "WARN ") + c["label"])
cfg = yaml.safe_load(open(X/"config/config.yaml"))

In [3]:
# --- recipe echo: config + the exact lines of integrate.py that implement it ---
print(json.dumps({k: cfg[k] for k in ("scvi", "leiden") if k in cfg}, indent=1))
src = open(X/"workflow/scripts/integrate.py").read()
for pat in (r"n_top_genes\s*=", r"exclude|mito|ribo", r"n_latent", r"n_layers", r"batch_key",
            r"n_neighbors", r"leiden", r"random_state|seed"):
    hits = [l.strip() for l in src.splitlines() if re.search(pat, l) and not l.strip().startswith("#")][:3]
    for h in hits: print(f"  [{pat[:14]:14s}] {h[:130]}")
check("config scvi.n_hvg", cfg["scvi"].get("n_hvg") == 3000, cfg["scvi"].get("n_hvg"), 3000)
check("config scvi.exclude_mito_ribo", cfg["scvi"].get("exclude_mito_ribo") is True, cfg["scvi"].get("exclude_mito_ribo"), True)
check("config scvi.n_latent", cfg["scvi"].get("n_latent") == 30, cfg["scvi"].get("n_latent"), 30)
check("config leiden.knn", cfg["leiden"].get("knn") == 100, cfg["leiden"].get("knn"), 100)

{
 "scvi": {
  "n_hvg": 3000,
  "hvg_flavor": "seurat_v3",
  "n_latent": 30,
  "n_layers": 2,
  "batch_key": "sample",
  "exclude_mito_ribo": true
 },
 "leiden": {
  "resolution": 0.1,
  "knn": 100
 }
}
  [n_top_genes\s*] 2. HVG selection: seurat_v3, n_top_genes=3000, batch_key='sample', on the raw
  [n_top_genes\s*] n_top_genes=N_HVG,
  [exclude|mito|r] 1. Label mito/ribo genes; split off a mito/ribo-EXCLUDED object for HVG selection.
  [exclude|mito|r] EXCLUDE_MITO_RIBO = scvi_cfg.get("exclude_mito_ribo", True)
  [exclude|mito|r] mito_mask = adata.var_names.str.startswith(("mt-", "Mt-", "MT-"))
  [n_latent      ] SCVI(n_layers=2, n_latent=30, gene_likelihood='nb'); train(early_stopping=True).
  [n_latent      ] N_LATENT = scvi_cfg.get("n_latent", 30)
  [n_latent      ] adata_hvg, n_layers=N_LAYERS, n_latent=N_LATENT,
  [n_layers      ] SCVI(n_layers=2, n_latent=30, gene_likelihood='nb'); train(early_stopping=True).
  [n_layers      ] N_LAYERS = scvi_cfg.get("n_layers", 2)
  [n_layers

In [4]:
# --- object-side verification per species ---
OBJ = {sp: X/f"data/interim/latent_{sp}.h5ad" for sp in ("mouse", "human")}
info = {}
for sp, p in OBJ.items():
    with h5py.File(p) as f:
        n = f["obs/_index"].shape[0]
        obsm = sorted(f["obsm"].keys()); uns = sorted(f["uns"].keys())
        lat = read_elem(f["obsm/X_scVI"]); um = read_elem(f["obsm/X_umap_scVI"])
        var_names = pd.Index(read_elem(f["var/_index"]))
        hv = read_elem(f["var/highly_variable"]) if "highly_variable" in f["var"] else None
        nb100 = read_elem(f["uns/neighbors_scVI_100"]) if "neighbors_scVI_100" in f["uns"] else None
        ladder_cols = [c for c in f["obs"].keys() if c.startswith("leiden_scVI_")]
        samples = pd.Series(read_elem(f["obs/sample"])).astype(str)
    info[sp] = dict(n=n, lat=lat, um=um, samples=samples, ladder=len(ladder_cols))
    check(f"{sp}: latent is {n} x 30", lat.shape == (n, 30), lat.shape)
    mito = var_names.str.startswith(("mt-", "Mt-", "MT-"))
    ribo = var_names.str.startswith(("Rps","Rpl","Mrps","Mrpl","RPS","RPL","MRPS","MRPL"))
    if hv is not None:
        hv = np.asarray(hv, dtype=bool)
        check(f"{sp}: n HVG == 3000", int(hv.sum()) == 3000, int(hv.sum()), 3000)
        check(f"{sp}: zero mito/ribo genes among HVGs", int((hv & (mito | ribo)).sum()) == 0,
              int((hv & (mito | ribo)).sum()), 0)
    else:
        check(f"{sp}: highly_variable flag stored on object", False, "missing", warn=True)
    if nb100 is not None:
        pr = nb100.get("params", {})
        check(f"{sp}: k=100 graph params", pr.get("n_neighbors") == 100 and pr.get("use_rep") == "X_scVI", pr)
    check(f"{sp}: leiden ladder present", info[sp]["ladder"] >= 10, info[sp]["ladder"], ">=10 resolutions")

PASS mouse: latent is 68069 x 30: got=(68069, 30)
PASS mouse: n HVG == 3000: got=3000 want=3000
PASS mouse: zero mito/ribo genes among HVGs: got=0 want=0
PASS mouse: k=100 graph params: got={'method': 'umap', 'metric': 'euclidean', 'n_neighbors': 100, 'random_state': 0, 'use_rep': 'X_scVI'}
PASS mouse: leiden ladder present: got=21 want=>=10 resolutions


PASS human: latent is 70363 x 30: got=(70363, 30)
PASS human: n HVG == 3000: got=3000 want=3000
PASS human: zero mito/ribo genes among HVGs: got=0 want=0
PASS human: k=100 graph params: got={'method': 'umap', 'metric': 'euclidean', 'n_neighbors': 100, 'random_state': 0, 'use_rep': 'X_scVI'}
PASS human: leiden ladder present: got=21 want=>=10 resolutions


In [5]:
# --- deliverable identity: netscratch atlas + fib/dec latents are row-subsets of THESE latents ---
N = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
PAIRS = [("mouse", N/"2026_08_28_Cevrim_XMens_All_Cells.h5ad", "X_scVI"),
         ("mouse", N/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad", "X_scVI"),
         ("human", N/"2026_08_28_Cevrim_Human_All_Cells.h5ad", "X_scVI_wholetissue"),
         ("human", N/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad", "X_scVI_wholetissue")]
for sp, p, key in PAIRS:
    if not p.exists():
        check(f"{p.name} exists", False, "missing", warn=True); continue
    with h5py.File(OBJ[sp]) as f:
        ai = pd.Index(read_elem(f["obs/_index"]))
    with h5py.File(p) as f:
        if key not in f["obsm"]:
            check(f"{p.name}: {key} present", False, sorted(f["obsm"].keys()), warn=True); continue
        B = read_elem(f[f"obsm/{key}"]); bi = pd.Index(read_elem(f["obs/_index"]))
    pos = ai.get_indexer(bi); ok = pos >= 0
    d = float(np.abs(B[ok] - info[sp]["lat"][pos[ok]]).max()) if ok.sum() else np.inf
    check(f"{p.name}[{key}] == latent_{sp} subset (max diff)", d == 0.0, d, 0.0)

FAIL 2026_08_28_Cevrim_XMens_All_Cells.h5ad[X_scVI] == latent_mouse subset (max diff): got=11.476592063903809 want=0.0


FAIL 2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad[X_scVI] == latent_mouse subset (max diff): got=10.058606147766113 want=0.0


WARN 2026_08_28_Cevrim_Human_All_Cells.h5ad: X_scVI_wholetissue present: got=['X_scVI', 'X_spatial', 'X_umap_scVI']


FAIL 2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad[X_scVI_wholetissue] == latent_human subset (max diff): got=6.883294105529785 want=0.0


In [6]:
# --- stored UMAPs, colored by sample (batch mixing, visual) — NO recomputation ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
rng = np.random.default_rng(0)
for ax, sp in zip(axes, ("mouse", "human")):
    um, sm = info[sp]["um"], info[sp]["samples"]
    idx = rng.permutation(len(um))[:60000]
    cats = sorted(sm.unique()); lut = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}
    ax.scatter(um[idx, 0], um[idx, 1], s=1.2, c=[lut[s] for s in sm.iloc[idx]], linewidths=0, rasterized=True)
    ax.set_title(f"latent_{sp}: stored atlas UMAP by sample (n={info[sp]['n']:,})", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(handles=[plt.Line2D([], [], ls="", marker="o", ms=4, color=lut[c], label=c) for c in cats],
              fontsize=5, frameon=False, loc="best", ncol=2)
fig.savefig("audit_02_umaps_by_sample.png", dpi=150, bbox_inches="tight")
print("wrote audit_02_umaps_by_sample.png")

wrote audit_02_umaps_by_sample.png


In [7]:
summary("02_integrate")


===== 18 checks | 3 FAIL | 1 WARN =====
FAIL 2026_08_28_Cevrim_XMens_All_Cells.h5ad[X_scVI] == latent_mouse subset (max diff)
FAIL 2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad[X_scVI] == latent_mouse subset (max diff)
FAIL 2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad[X_scVI_wholetissue] == latent_human subset (max diff)
WARN 2026_08_28_Cevrim_Human_All_Cells.h5ad: X_scVI_wholetissue present
